# PCA & FDA/LDA Analysis -- Irish Home BER Dataset

Full dimensionality-reduction pipeline:
1. Pre-PCA Feature Audit
2. Scaling
3. PCA Analysis
4. FDA / LDA Analysis
5. PCA vs LDA Comparison
6. Recommendations


In [8]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

DATA_FILE  = 'BER_Cleaned_Optimized.parquet'
TARGET_COL = 'BerRating'
FDA_LABEL  = 'DwellingTypeDescr'
DROP_FROM_FEATURES = ['CountyName', 'DwellingTypeDescr', 'BerRating']

print("Libraries loaded.")

Libraries loaded.


## Step 1 - Pre-PCA Feature Audit

Check categorical variance, flag near-zero-variance columns, decide encoding strategy.

In [9]:
print("Loading dataset ...")
df_raw = pd.read_parquet(DATA_FILE)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols")
print(f"Missing values: {df_raw.isnull().sum().sum()}")

cat_cols   = df_raw.select_dtypes(include='category').columns.tolist()
float_cols = df_raw.select_dtypes(include=['float32','float64']).columns.tolist()

print(f"\n13 categorical columns: {cat_cols}")

NZV_THRESHOLD = 0.95
nzv_flags = []
print("\n--- Categorical Value Counts (top-1 share) ---")
for col in cat_cols:
    vc = df_raw[col].value_counts(normalize=True)
    top_share = vc.iloc[0]
    n_unique  = vc.shape[0]
    flag = "NZV FLAG" if top_share >= NZV_THRESHOLD else "OK"
    if top_share >= NZV_THRESHOLD:
        nzv_flags.append(col)
    print(f"  {col:<30s}  top={top_share:.1%}  unique={n_unique:>3d}  {flag}")

print(f"\nNear-zero-variance categoricals (>={NZV_THRESHOLD:.0%}) -> drop before PCA:")
for c in nzv_flags:
    print(f"   {c}")

print("\n--- Float near-zero-variance check ---")
float_nzv = []
for col in float_cols:
    if col in DROP_FROM_FEATURES:
        continue
    std = df_raw[col].std()
    if std < 1e-6:
        float_nzv.append(col)
        print(f"  {col} is near-constant (std={std:.2e}) -> DROP")
if not float_nzv:
    print("  All float columns have meaningful variance. OK")

print("\n--- Encoding Strategy ---")
yes_no_cols   = []
multicat_cols = []
ignore_cols   = DROP_FROM_FEATURES + nzv_flags

for col in cat_cols:
    if col in ignore_cols:
        continue
    uniq = set(str(v).upper() for v in df_raw[col].dropna().unique())
    if uniq <= {'YES', 'NO'}:
        yes_no_cols.append(col)
    else:
        multicat_cols.append(col)

print(f"  Binary (0/1):   {yes_no_cols}")
print(f"  One-hot:        {multicat_cols}")
print(f"  Dropped (geo/label/NZV): {ignore_cols}")

Loading dataset ...
Shape: 1,351,582 rows x 45 cols
Missing values: 0

13 categorical columns: ['CountyName', 'DwellingTypeDescr', 'VentilationMethod', 'StructureType', 'SuspendedWoodenFloor', 'CHBoilerThermostatControlled', 'OBBoilerThermostatControlled', 'OBPumpInsideDwelling', 'WarmAirHeatingSystem', 'UndergroundHeating', 'CylinderStat', 'CombinedCylinder', 'ThermalMassCategory']

--- Categorical Value Counts (top-1 share) ---
  CountyName                      top=9.6%  unique= 55  OK
  DwellingTypeDescr               top=30.2%  unique= 11  OK
  VentilationMethod               top=87.8%  unique=  6  OK
  StructureType                   top=88.8%  unique=  4  OK
  SuspendedWoodenFloor            top=92.6%  unique=  3  OK
  CHBoilerThermostatControlled    top=52.4%  unique=  2  OK
  OBBoilerThermostatControlled    top=90.4%  unique=  2  OK
  OBPumpInsideDwelling            top=93.0%  unique=  2  OK
  WarmAirHeatingSystem            top=99.7%  unique=  2  NZV FLAG
  UndergroundHeating 

## Step 2 - Encoding & Scaling

In [10]:
df = df_raw.copy()

# Binary encode YES/NO
for col in yes_no_cols:
    df[col] = df[col].astype(str).str.upper().map({'YES': 1, 'NO': 0})

# One-hot encode multi-class cats
df = pd.get_dummies(df, columns=multicat_cols, drop_first=False)

# Drop unwanted columns
drop_set = set(nzv_flags + float_nzv + DROP_FROM_FEATURES)
feature_cols = [c for c in df.columns if c not in drop_set]

X = df[feature_cols].astype(np.float32).values
y_rating = df_raw[TARGET_COL].values
y_label  = df_raw[FDA_LABEL].astype(str).values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix after encoding+scaling: {X_scaled.shape}")
print(f"Feature count: {X_scaled.shape[1]}")

Feature matrix after encoding+scaling: (1351582, 57)
Feature count: 57


## Step 3 - PCA Analysis

In [11]:
print("Running full PCA ...")
pca    = PCA()
X_pca  = pca.fit_transform(X_scaled)
ev_ratio = pca.explained_variance_ratio_
ev_cum   = np.cumsum(ev_ratio)

for threshold in [0.80, 0.90, 0.95]:
    n_comp = int(np.searchsorted(ev_cum, threshold)) + 1
    print(f"  Components for {threshold:.0%} variance: {n_comp}")

print("\n--- Cumulative Variance (first 20 PCs) ---")
print(f"{'PC':>4}  {'ExplVar%':>9}  {'CumVar%':>9}")
for i in range(min(20, len(ev_ratio))):
    print(f"  PC{i+1:>2d}  {ev_ratio[i]*100:>8.2f}%  {ev_cum[i]*100:>8.2f}%")

# --- Scree plot ---
n_show = 40
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, n_show+1), ev_ratio[:n_show]*100, color='steelblue', alpha=0.8)
axes[0].set(xlabel='Principal Component', ylabel='Explained Variance (%)', title='Scree Plot')
axes[1].plot(range(1, n_show+1), ev_cum[:n_show]*100, marker='o', markersize=3, color='steelblue')
axes[1].axhline(90, color='orange', linestyle='--', label='90%')
axes[1].axhline(95, color='red',    linestyle='--', label='95%')
axes[1].set(xlabel='Number of Components', ylabel='Cumulative Explained Variance (%)', title='Cumulative Variance')
axes[1].legend()
plt.tight_layout()
plt.savefig('pca_scree.png', dpi=150)
plt.close()
print("Saved: pca_scree.png")

# --- PC1 vs PC2 scatter ---
sample_idx = np.random.choice(len(X_pca), size=min(20000, len(X_pca)), replace=False)
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(X_pca[sample_idx, 0], X_pca[sample_idx, 1],
                c=y_rating[sample_idx], cmap='RdYlGn', alpha=0.3, s=3)
plt.colorbar(sc, ax=ax, label='BerRating')
ax.set(xlabel='PC1', ylabel='PC2', title='PC1 vs PC2 -- colored by BerRating (sample 20k)')
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=150)
plt.close()
print("Saved: pca_scatter.png")

# --- Biplot ---
loadings = pd.DataFrame(pca.components_.T, index=feature_cols,
                         columns=[f'PC{i+1}' for i in range(pca.n_components_)])
top10_pc1 = loadings['PC1'].abs().nlargest(10).index
scale_x = 1.0 / (X_pca[:, 0].max() - X_pca[:, 0].min())
scale_y = 1.0 / (X_pca[:, 1].max() - X_pca[:, 1].min())

fig, ax = plt.subplots(figsize=(11, 8))
ax.scatter(X_pca[sample_idx, 0]*scale_x, X_pca[sample_idx, 1]*scale_y,
           c=y_rating[sample_idx], cmap='RdYlGn', alpha=0.15, s=3)
for feat in top10_pc1:
    idx = feature_cols.index(feat)
    ax.annotate('', xy=(pca.components_[0, idx], pca.components_[1, idx]),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='navy', lw=1.5))
    ax.text(pca.components_[0, idx]*1.05, pca.components_[1, idx]*1.05,
            feat, fontsize=7, color='navy')
ax.set(xlabel='PC1', ylabel='PC2', title='Biplot -- Top 10 PC1 Features')
plt.tight_layout()
plt.savefig('pca_biplot.png', dpi=150)
plt.close()
print("Saved: pca_biplot.png")

# --- Heatmap ---
top15_overall = loadings.iloc[:, :5].abs().max(axis=1).nlargest(15).index
heat_data = loadings.loc[top15_overall, ['PC1','PC2','PC3','PC4','PC5']]
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Top 15 Feature Loadings -- PC1 to PC5')
plt.tight_layout()
plt.savefig('pca_heatmap.png', dpi=150)
plt.close()
print("Saved: pca_heatmap.png")

# --- PC-BerRating correlation ---
print("\n--- Correlation of each PC with BerRating (PC1-PC10) ---")
for i in range(min(10, X_pca.shape[1])):
    corr = np.corrcoef(X_pca[:, i], y_rating)[0, 1]
    print(f"  PC{i+1:>2d}: r = {corr:+.4f}")

# --- Top 5 features per PC ---
print("\n--- Top 5 Contributing Features per PC ---")
for pc in ['PC1','PC2','PC3','PC4','PC5']:
    top5 = loadings[pc].abs().nlargest(5)
    print(f"  {pc}: {list(zip(top5.index, top5.values.round(3)))}")

Running full PCA ...
  Components for 80% variance: 24
  Components for 90% variance: 31
  Components for 95% variance: 37

--- Cumulative Variance (first 20 PCs) ---
  PC   ExplVar%    CumVar%
  PC 1     13.61%     13.61%
  PC 2      9.41%     23.02%
  PC 3      5.30%     28.32%
  PC 4      4.65%     32.97%
  PC 5      4.25%     37.22%
  PC 6      4.04%     41.27%
  PC 7      3.40%     44.67%
  PC 8      3.36%     48.03%
  PC 9      3.12%     51.14%
  PC10      2.85%     53.99%
  PC11      2.54%     56.53%
  PC12      2.44%     58.97%
  PC13      2.30%     61.27%
  PC14      2.16%     63.43%
  PC15      2.00%     65.43%
  PC16      1.89%     67.32%
  PC17      1.80%     69.12%
  PC18      1.78%     70.90%
  PC19      1.77%     72.67%
  PC20      1.76%     74.43%
Saved: pca_scree.png
Saved: pca_scatter.png
Saved: pca_biplot.png
Saved: pca_heatmap.png

--- Correlation of each PC with BerRating (PC1-PC10) ---
  PC 1: r = -0.6442
  PC 2: r = -0.2118
  PC 3: r = -0.1344
  PC 4: r = +0.2680

## Step 4 - FDA / LDA Analysis

In [12]:
print("Running LDA ...")
le    = LabelEncoder()
y_enc = le.fit_transform(y_label)
n_classes         = len(le.classes_)
n_components_lda  = min(n_classes - 1, X_scaled.shape[1])

lda   = LDA(n_components=n_components_lda)
X_lda = lda.fit_transform(X_scaled, y_enc)

print(f"Classes: {list(le.classes_)}")
print(f"LDA components: {X_lda.shape[1]}")
print(f"Explained variance ratio: {lda.explained_variance_ratio_.round(3)}")

# --- LD1 vs LD2 scatter ---
palette = plt.cm.get_cmap('tab10', n_classes)
fig, ax = plt.subplots(figsize=(12, 8))
for i, cls in enumerate(le.classes_):
    mask = np.where(y_enc == i)[0]
    s_mask = np.random.choice(mask, size=min(3000, len(mask)), replace=False)
    ld2_vals = X_lda[s_mask, 1] if X_lda.shape[1] > 1 else np.zeros(len(s_mask))
    ax.scatter(X_lda[s_mask, 0], ld2_vals, color=palette(i), label=cls, alpha=0.4, s=5)
ax.set(xlabel='LD1', ylabel='LD2', title='LDA -- LD1 vs LD2 by DwellingTypeDescr (sample)')
ax.legend(fontsize=7, markerscale=3, loc='upper right')
plt.tight_layout()
plt.savefig('lda_scatter.png', dpi=150)
plt.close()
print("Saved: lda_scatter.png")

# --- Explained variance bar ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(1, len(lda.explained_variance_ratio_)+1),
       lda.explained_variance_ratio_*100, color='teal', alpha=0.8)
ax.set(xlabel='LDA Component', ylabel='Explained Variance (%)', title='LDA Explained Variance Ratio')
plt.tight_layout()
plt.savefig('lda_variance.png', dpi=150)
plt.close()
print("Saved: lda_variance.png")

# --- Top discriminating features ---
lda_coef_df = pd.DataFrame(lda.coef_, columns=feature_cols)
print("\n--- Top 5 discriminating features for LD1 ---")
ld1_top = lda_coef_df.iloc[0].abs().nlargest(5)
print(list(zip(ld1_top.index, ld1_top.values.round(3))))
if lda_coef_df.shape[0] > 1:
    print("--- Top 5 discriminating features for LD2 ---")
    ld2_top = lda_coef_df.iloc[1].abs().nlargest(5)
    print(list(zip(ld2_top.index, ld2_top.values.round(3))))

Running LDA ...
Classes: ['Apartment', 'Basement Dwelling', 'Detached house', 'End of terrace house', 'Ground-floor apartment', 'House', 'Maisonette', 'Mid-floor apartment', 'Mid-terrace house', 'Semi-detached house', 'Top-floor apartment']
LDA components: 10
Explained variance ratio: [0.523 0.258 0.161 0.028 0.012 0.008 0.007 0.002 0.001 0.   ]
Saved: lda_scatter.png
Saved: lda_variance.png

--- Top 5 discriminating features for LD1 ---
[('CHBoilerThermostatControlled_NO ', np.float32(600.216)), ('CHBoilerThermostatControlled_YES', np.float32(599.693)), ('UndergroundHeating_NO ', np.float32(247.65)), ('UndergroundHeating_YES', np.float32(246.628)), ('StructureType_Masonry                       ', np.float32(150.352))]
--- Top 5 discriminating features for LD2 ---
[('CHBoilerThermostatControlled_NO ', np.float32(495.764)), ('CHBoilerThermostatControlled_YES', np.float32(495.09)), ('SuspendedWoodenFloor_No                            ', np.float32(146.393)), ('SuspendedWoodenFloor_Yes (U

## Step 5 - PCA vs LDA vs Original: Classification Accuracy (5-fold CV)

In [13]:
print("Running 5-fold CV comparison (sub-sampled to 100k rows) ...")

rng    = np.random.default_rng(42)
idx_cv = rng.choice(len(X_scaled), size=min(100_000, len(X_scaled)), replace=False)
X_cv   = X_scaled[idx_cv]
y_cv   = y_enc[idx_cv]

n90      = int(np.searchsorted(ev_cum, 0.90)) + 1
X_pca90  = X_pca[idx_cv, :n90]
X_lda_cv = X_lda[idx_cv]

CV      = 5
results = {}
label_map = {
    'Original':               X_cv,
    f'PCA (n={n90})':         X_pca90,
    f'LDA (n={X_lda_cv.shape[1]})': X_lda_cv,
}

for label, Xf in label_map.items():
    lr_scores = cross_val_score(
        LogisticRegression(max_iter=500, random_state=42),
        Xf, y_cv, cv=CV, scoring='accuracy', n_jobs=-1)
    rf_scores = cross_val_score(
        RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
        Xf, y_cv, cv=CV, scoring='accuracy')
    results[label] = {
        'LogReg Acc': f"{lr_scores.mean():.4f} +/- {lr_scores.std():.4f}",
        'RF Acc':     f"{rf_scores.mean():.4f} +/- {rf_scores.std():.4f}"
    }
    print(f"  {label}: LogReg={lr_scores.mean():.4f}, RF={rf_scores.mean():.4f}")

print("\n=== COMPARISON TABLE ===")
df_results = pd.DataFrame(results).T
print(df_results.to_string())

Running 5-fold CV comparison (sub-sampled to 100k rows) ...
  Original: LogReg=0.7809, RF=0.7784
  PCA (n=31): LogReg=0.6530, RF=0.6225
  LDA (n=10): LogReg=0.7564, RF=0.7763

=== COMPARISON TABLE ===
                   LogReg Acc             RF Acc
Original    0.7809 +/- 0.0011  0.7784 +/- 0.0022
PCA (n=31)  0.6530 +/- 0.0014  0.6225 +/- 0.0006
LDA (n=10)  0.7564 +/- 0.0019  0.7763 +/- 0.0017


## Step 6 - Recommendations

In [14]:
n80 = int(np.searchsorted(ev_cum, 0.80)) + 1
n90 = int(np.searchsorted(ev_cum, 0.90)) + 1
n95 = int(np.searchsorted(ev_cum, 0.95)) + 1

print("=== FINAL RECOMMENDATIONS ===")
print()
print("FEATURE ENGINEERING:")
print(f"  Drop near-zero-variance categoricals: {nzv_flags}")
print(f"  Drop float NZV columns:               {float_nzv if float_nzv else 'None found'}")
print("  Drop CountyName (geographic identifier, not building feature)")
print()
print("DIMENSIONALITY REDUCTION:")
print(f"  PCA:  {n80} components -> 80% variance")
print(f"  PCA:  {n90} components -> 90% variance  <-- RECOMMENDED for regression")
print(f"  PCA:  {n95} components -> 95% variance")
print(f"  LDA:  {n_components_lda} components (class-driven, best for classification)")
print()
print("DOWNSTREAM MODELLING:")
print(f"  BerRating regression     -> use PCA n={n90} features")
print("  DwellingType classification -> use LDA features (better cluster separation)")
print("  Random Forest baseline   -> original features viable (tree-based, scale-free)")
print()
print("PLOTS SAVED:")
print("  pca_scree.png, pca_scatter.png, pca_biplot.png, pca_heatmap.png")
print("  lda_scatter.png, lda_variance.png")

=== FINAL RECOMMENDATIONS ===

FEATURE ENGINEERING:
  Drop near-zero-variance categoricals: ['WarmAirHeatingSystem', 'CylinderStat', 'CombinedCylinder']
  Drop float NZV columns:               None found
  Drop CountyName (geographic identifier, not building feature)

DIMENSIONALITY REDUCTION:
  PCA:  24 components -> 80% variance
  PCA:  31 components -> 90% variance  <-- RECOMMENDED for regression
  PCA:  37 components -> 95% variance
  LDA:  10 components (class-driven, best for classification)

DOWNSTREAM MODELLING:
  BerRating regression     -> use PCA n=31 features
  DwellingType classification -> use LDA features (better cluster separation)
  Random Forest baseline   -> original features viable (tree-based, scale-free)

PLOTS SAVED:
  pca_scree.png, pca_scatter.png, pca_biplot.png, pca_heatmap.png
  lda_scatter.png, lda_variance.png


Exception ignored in: <function ResourceTracker.__del__ at 0x105eedda0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10a0edda0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x103205da0>
Traceback (most recent call last

In [19]:
import io
buffer = io.StringIO()
buffer.write("=== FINAL RECOMMENDATIONS ===")
buffer.write("\n")
buffer.write("FEATURE ENGINEERING:\n")
buffer.write(f"  Drop near-zero-variance categoricals: {nzv_flags}\n")
buffer.write(f"  Drop float NZV columns:               {float_nzv if float_nzv else 'None found'}\n")
buffer.write("  Drop CountyName (geographic identifier, not building feature)\n")
buffer.write("\n")
buffer.write("DIMENSIONALITY REDUCTION:\n")
buffer.write(f"  PCA:  {n80} components -> 80% variance\n")
buffer.write(f"  PCA:  {n90} components -> 90% variance  <-- RECOMMENDED for regression\n")
buffer.write(f"  PCA:  {n95} components -> 95% variance\n")
buffer.write(f"  LDA:  {n_components_lda} components (class-driven, best for classification)\n")
buffer.write("\n")
buffer.write("DOWNSTREAM MODELLING:\n")
buffer.write(f"  BerRating regression     -> use PCA n={n90} features\n")
buffer.write("  DwellingType classification -> use LDA features (better cluster separation)\n")
buffer.write("  Random Forest baseline   -> original features viable (tree-based, scale-free)\n")
buffer.write("\n")
buffer.write("PLOTS SAVED:\n")
buffer.write("  pca_scree.png, pca_scatter.png, pca_biplot.png, pca_heatmap.png\n")
buffer.write("  lda_scatter.png, lda_variance.png\n")
info_buf = io.StringIO()
buffer.write(info_buf.getvalue())

summary_content = buffer.getvalue()
print(summary_content)
# Save to file
summary_file = 'pca_summary_output.txt'
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write(summary_content)

print(f"\nValidation summary saved to: {summary_file}")

=== FINAL RECOMMENDATIONS ===
FEATURE ENGINEERING:
  Drop near-zero-variance categoricals: ['WarmAirHeatingSystem', 'CylinderStat', 'CombinedCylinder']
  Drop float NZV columns:               None found
  Drop CountyName (geographic identifier, not building feature)

DIMENSIONALITY REDUCTION:
  PCA:  24 components -> 80% variance
  PCA:  31 components -> 90% variance  <-- RECOMMENDED for regression
  PCA:  37 components -> 95% variance
  LDA:  10 components (class-driven, best for classification)

DOWNSTREAM MODELLING:
  BerRating regression     -> use PCA n=31 features
  DwellingType classification -> use LDA features (better cluster separation)
  Random Forest baseline   -> original features viable (tree-based, scale-free)

PLOTS SAVED:
  pca_scree.png, pca_scatter.png, pca_biplot.png, pca_heatmap.png
  lda_scatter.png, lda_variance.png


Validation summary saved to: pca_summary_output.txt
